# Colab SSH Setup

1. **Runtime → Change runtime type → T4 GPU**
2. **Run all cells** (Runtime → Run all)
3. Copy the SSH config from the output and add to `~/.ssh/config`
4. Connect: `ssh colab`

After connecting:
```bash
cd /content/Tubitak-2209B
tmux new -s train
bash scripts/train_all_models.sh
```

In [ ]:
# Mount Google Drive and extract dataset
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess

TAR_PATH = "/content/drive/MyDrive/Tubitak-2209B/data_split.tar.gz"
LOCAL_DATA = "/content/data/processed_merged_split"

if not os.path.exists(LOCAL_DATA):
    print("Extracting data (~5-10 min)...")
    os.makedirs("/content/data", exist_ok=True)
    subprocess.run(["tar", "-xzf", TAR_PATH, "-C", "/content/data"], check=True)
    print("Done!")
else:
    print(f"Data already exists: {LOCAL_DATA}")

# Verify
for split in ["train", "val", "test"]:
    split_dir = os.path.join(LOCAL_DATA, split)
    if os.path.exists(split_dir):
        total = sum(len(os.listdir(os.path.join(split_dir, c))) for c in os.listdir(split_dir))
        print(f"  {split}: {total} images")

In [ ]:
# Install cloudflared and start SSH tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Set root password for SSH
import subprocess
subprocess.run(["bash", "-c", "echo 'root:tubitak2209' | chpasswd"], check=True)

# Configure and start SSH server
!apt-get -qq install openssh-server > /dev/null 2>&1
!echo "PermitRootLogin yes" >> /etc/ssh/sshd_config
!echo "PasswordAuthentication yes" >> /etc/ssh/sshd_config
!service ssh start

# Start cloudflared tunnel in background
import threading, re, time

tunnel_url = None

def run_tunnel():
    global tunnel_url
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "ssh://localhost:22", "--no-autoupdate"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(line.strip())
        match = re.search(r'https://(.+\.trycloudflare\.com)', line)
        if match:
            tunnel_url = match.group(1)

t = threading.Thread(target=run_tunnel, daemon=True)
t.start()

# Wait for tunnel URL
for _ in range(30):
    if tunnel_url:
        break
    time.sleep(1)

if tunnel_url:
    print(f"\n{'='*60}")
    print(f"SSH TUNNEL READY!")
    print(f"{'='*60}")
    print(f"\nAdd to ~/.ssh/config:")
    print(f"""\nHost colab\n    HostName {tunnel_url}\n    User root\n    ProxyCommand cloudflared access ssh --hostname %h\n""")
    print(f"Then connect: ssh colab")
    print(f"Password: tubitak2209")
else:
    print("ERROR: Tunnel failed to start")

In [ ]:
# Clone repo, install dependencies, and set up tmux
import os
os.chdir("/content")

!git clone https://github.com/halitartun/Tubitak-2209B.git 2>/dev/null || (cd Tubitak-2209B && git pull)
!cd Tubitak-2209B && pip install -r requirements-train.txt -q

# Install tmux for persistent sessions
!apt-get -qq install tmux > /dev/null 2>&1

print("\nSetup complete! After SSH:")
print("  cd /content/Tubitak-2209B")
print("  tmux new -s train")
print("  bash scripts/train_all_models.sh")

In [ ]:
# Keep-alive cell — prevents Colab idle timeout
import time

print("Keep-alive active. Don't close this tab.")
while True:
    time.sleep(300)
    print(f"Still alive... {time.strftime('%H:%M:%S')}")